In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "transactions.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sergionefedov/fraud-detection-1m-transactions-7-fraud-types",
  file_path,
)

/tmp/ipykernel_19986/3901280428.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 122M/122M [00:07<00:00, 17.3MB/s]


## Importation des bibliothèques

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from google.colab import drive
import joblib
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE_PATH   = '/content/drive/MyDrive/fraud_detection'
DATA_PATH   = f'{BASE_PATH}/data'
MODELS_PATH = f'{BASE_PATH}/models'

os.makedirs(DATA_PATH,   exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

## Données cibles

In [4]:
y_detection      = df['is_fraud']
y_classification = df['fraud_pattern']

## Suppression des colonnes

In [5]:
cols_a_supprimer = [
    'transaction_id',
    'account_id',
    'timestamp',
    'is_weekend',
    'amount',
    'is_fraud',
    'fraud_pattern',
]

df_clean = df.drop(columns=cols_a_supprimer)

## Encodage des variables catégorielles

In [10]:
cols_categorielle = ['merchant_category', 'merchant_country', 'device_type']
cols_numerique    = [col for col in df_clean.columns if col not in cols_categorielle]

X_num = df_clean[cols_numerique].values
X_cat = df_clean[cols_categorielle].values

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X_cat)

In [11]:
X = np.hstack([X_num, X_cat_encoded])

## Création des jeux de données (Celui contenant les cas normaux, celui contenant les fraudes, types de fraudes)

In [13]:
X_normal = X[y_detection.values == 0]
X_fraude  = X[y_detection.values == 1]
y_fraude  = y_classification[y_detection.values == 1].values

## Standardisation des valeurs

In [23]:
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_fraude_scaled = scaler.transform(X_fraude)
X_scaled    = scaler.transform(X)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Sauvegarde des jeux de données

In [24]:
# Dataset standardisé complet + variable cible
np.save(f'{DATA_PATH}/X_data.npy', X_scaled)
np.save(f'{DATA_PATH}/y_detection.npy', y_detection.values)

# Dataset normales (entraînement AE et IF)
np.save(f'{DATA_PATH}/X_normal.npy', X_normal_scaled)

# Dataset fraudes (classification)
np.save(f'{DATA_PATH}/X_fraude.npy', X_fraude_scaled)
np.save(f'{DATA_PATH}/y_fraude.npy', y_fraude)

# Modèles de preprocessing
joblib.dump(encoder, f'{MODELS_PATH}/onehot_fraud_encoder.pkl')
joblib.dump(scaler, f'{MODELS_PATH}/fraud_scaler.pkl')

['/content/drive/MyDrive/fraud_detection/models/fraud_scaler.pkl']